Defining the Feature Extractor


In [ ]:
from transformers import ASTFeatureExtractor, ASTForAudioClassification

feature_extractor = ASTFeatureExtractor.from_pretrained(
    "MIT/ast-finetuned-audioset-10-10-0.4593"
)

Hardware Verification

In [3]:
car_diagnostic_labels = [
    "bad_ignition",
    "cv joint",
    "fuel pump",
    "amortiseur",
    "serpentine belt",
    "tie road",
    "worn_out_brakes"
]
label2id = {label: i for i, label in enumerate(car_diagnostic_labels)}
id2label = {i: label for i, label in enumerate(car_diagnostic_labels)}

Defining the Audio Model


In [4]:
model = ASTForAudioClassification.from_pretrained(
    "MIT/ast-finetuned-audioset-10-10-0.4593",
    num_labels=7,
    ignore_mismatched_sizes=True
)
model.config.label2id = label2id
model.config.id2label = id2label
model.config.num_labels = len(car_diagnostic_labels)
for param in model.parameters():
    param.requires_grad = False

for param in model.classifier.parameters():
    param.requires_grad = True


Some weights of ASTForAudioClassification were not initialized from the model checkpoint at MIT/ast-finetuned-audioset-10-10-0.4593 and are newly initialized because the shapes did not match:
- classifier.dense.bias: found shape torch.Size([527]) in the checkpoint and torch.Size([7]) in the model instantiated
- classifier.dense.weight: found shape torch.Size([527, 768]) in the checkpoint and torch.Size([7, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Defining the evaluation Metrics

In [5]:
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    
    # On définit 'macro', 'micro' ou 'weighted' pour le multiclasse
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, 
        preds, 
        average='macro'  # C'est ici que ça se joue !
    )
    acc = accuracy_score(labels, preds)
    
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

Training Data

In [6]:
from functions import get_Data
train_Data,test_data=get_Data("augmented-data",feature_extractor)

Getting data worn_out_brakes: 100%|██████████| 300/300 [00:01<00:00, 215.70it/s]


In [7]:

from transformers import TrainingArguments
training_args = TrainingArguments(
    output_dir="./new training",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=20,
    learning_rate=8e-6,
    remove_unused_columns=False,
    num_train_epochs=20,
    report_to="tensorboard", 
    
)  
from transformers import Trainer
trainer = Trainer(

    model=model,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=train_Data,
    eval_dataset=test_data,
  
)
    


In [8]:
from transformers import Trainer
trainer.train(resume_from_checkpoint=True)


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
9,1.028200,1.020952,0.771242,0.732950,0.818845,0.710352
10,0.958600,0.967300,0.797386,0.762938,0.835890,0.742495
11,0.954600,0.921848,0.816993,0.779836,0.827797,0.765709
12,0.850200,0.884045,0.830065,0.803680,0.843355,0.787687
13,0.846400,0.851525,0.836601,0.811008,0.848419,0.794830
14,0.817100,0.824860,0.843137,0.819356,0.845130,0.805869
15,0.821600,0.802946,0.849673,0.825793,0.849408,0.813012
16,0.750500,0.785807,0.849673,0.825793,0.849408,0.813012
17,0.736700,0.772925,0.849673,0.825502,0.848005,0.813012
18,0.797500,0.763739,0.843137,0.820422,0.837942,0.809765


TrainOutput(global_step=3760, training_loss=0.5057356606138513, metrics={'train_runtime': 4196.4895, 'train_samples_per_second': 14.321, 'train_steps_per_second': 0.896, 'total_flos': 4.0739304098758656e+18, 'train_loss': 0.5057356606138513, 'epoch': 20.0})